# 01 · 脑电中的传播顺序会重复吗？

**电压 → 群事件 → 通道先后顺序 → 模板 → 一致性统计**

用 E1146 的四段 200 秒原始 SEEG 走完 Fig1 的分析。每张图都来自下面重新检测的事件。


In [ ]:
import _setup
import numpy as np
from data_processing import raw_to_events as raw
from analysis import from_events as analysis
from plotting import computational as plot
from workflows.tutorial import raw_inputs, TutorialOutput, save_events

config, raw_root = raw_inputs()
out = TutorialOutput('fig1')


## Fig1A · 找到多个通道共同参与的事件

先减去参考信号、滤波，再检测各通道 HFO 并合并为群事件。下面显示第一个群事件的波形。


In [ ]:
record = config['records'][0]
samples, fs, names = raw.read_samples(raw_root/record['file'],
    start_sec=record.get('start_sec', 0), duration_sec=record['duration_sec'], channels=record.get('channels'))
pre = raw.preprocess(samples, fs, names, config)
detected = raw.detect_groups(pre, config)
out.show(plot.waveform(pre, detected), 'fig1_A_waveforms')


## Fig1B · 用什么时刻代表每个通道？

在事件窗内，以时频幅度的三次方 $S^3$ 为权重计算时间质心。图中竖线代表质心；比较它们就得到传播先后。


In [ ]:
features = raw.event_features(pre, detected)
out.show(plot.spectra(pre, features, index=0), 'fig1_B_centroids')


## Fig1C · 把每次事件写成一个顺序

将参与通道的质心从早到晚转换成 rank，再按通道名汇总各记录。每列是一个事件，空白是未参与的通道。


In [ ]:
features['block_ids'] = np.full(features['ranks'].shape[1], record['block_id'])
blocks = [features] + [raw.process_record(r, config, raw_root)[2] for r in config['records'][1:]]
events = raw.join_blocks(blocks)
out.show(plot.rank_heatmap(events), 'fig1_C_ranks')


## Fig1E · 哪些事件具有相似顺序？

用参与通道的归一化 rank 做 K-means，再按类别排列事件。这里设 K=2 来提取两个模板；它是分析设定，不代表已证明存在两个生理状态。


In [ ]:
clustered = analysis.cluster_events(events, k=2, seed=0)
out.show(plot.rank_heatmap(events, clustered), 'fig1_E_kmeans')


## Fig1D · 重复顺序是否超过随机基线？

Matching index 衡量共同参与通道对的先后是否一致。保持参与关系、打乱顺序，得到置换基线。


In [ ]:
stats = analysis.patient_statistics(events, clustered, permutations=200, seed=0)
out.show(plot.statistics(stats, panel='D'), 'fig1_D_statistics')


## Fig1F · 分组后，顺序是否更一致？

比较全部事件与各模板内部的 matching index。这里仍是同一位患者；完整队列需先得到每位患者的值，再做患者间统计。


In [ ]:
out.show(plot.statistics(stats, panel='F'), 'fig1_F_template_statistics')


## 下一步：这些顺序在空间中意味着什么？

把本次算出的事件交给 [Notebook 02](02_ranks_to_spatial_fields_fig2.ipynb)。更换原始数据、检测参数和运行完整队列的方法见 [方法说明](../METHODS_AND_PANELS.md#数据接口)。


In [ ]:
save_events(out, events, stats, config, raw_root)
